
# Registro do melhor modelo


## Setup

In [0]:
import mlflow
import pandas as pd
import numpy as np


## Identificação do modelo campeão

In [0]:
EXPERIMENT_NAME = '/Users/lul.rafaelbarbosa@gmail.com/reactivation-customers-experiment'
METRIC_NAME = 'test_pr_auc'

In [0]:
best_run_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="attributes.status = 'FINISHED'",
    order_by=[f"metrics.{METRIC_NAME} DESC"],
    max_results=1
)

best_run = best_run_df.iloc[0]

best_run_id = best_run["run_id"]
best_metric_value = best_run[f"metrics.{METRIC_NAME}"]

print(f"Run ID: {best_run_id}")
print(f"{METRIC_NAME}: {best_metric_value}")


## Definição dos Guardrails

In [0]:
GUARDRAILS_METRICS = {
    'metrics.test_pr_auc': 0.30,
    'metrics.test_lift_at_10pct': 2.5
}


## Executar a validação automática

In [0]:
guardrail_results = []

for metric_name, minimum_value in GUARDRAILS_METRICS.items():
    metric_value = best_run.get(metric_name, np.nan)

    metric_is_valid = pd.notna(metric_value) and np.isfinite(float(metric_value))

    approved = metric_is_valid and float(metric_value) >= minimum_value

    guardrail_results.append(
        {
            "guardrail": metric_name,
            "minimum_value": minimum_value,
            "actual_value": metric_value,
            "status": ("APPROVED" if approved else "BLOCKED"),
            "approved": approved,
        }
    )

In [0]:
guardrails_df = pd.DataFrame(guardrail_results)

display(guardrails_df)

In [0]:
promotion_approved = guardrails_df["approved"].all()

print(
    f"""
    Promotion status:

    {"APPROVED" if promotion_approved else "BLOCKED"}
    """
)

In [0]:
failed_guardrails = guardrails_df.query("approved == False")

if not promotion_approved:
    failed_metrics = failed_guardrails[
        [
            "guardrail",
            "minimum_value",
            "actual_value",
        ]
    ].to_dict(orient="records")

    raise ValueError(
        f"""
        Promoção bloqueada.

        Guardrails reprovados:

        {failed_metrics}
        """
    )

print(
    """
    Todos os guardrails foram aprovados.

    Modelo autorizado para registro.
    """
)


## Registro do modelo

In [0]:
REGISTERED_MODEL_NAME = "workspace.default.reactivation_customers_model"
MODEL_ARTIFACT_PATH = "model"
mlflow.set_registry_uri("databricks-uc")
model_uri = f"runs:/{best_run_id}/{MODEL_ARTIFACT_PATH}"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=(REGISTERED_MODEL_NAME),
)

model_version = str(registered_model.version)

print(
    f"""
    Modelo registrado com sucesso.

    Modelo:
    {REGISTERED_MODEL_NAME}

    Versão:
    {model_version}
    """
)


### Documentar a versão

In [0]:
client = mlflow.MlflowClient()

MODEL_DESCRIPTION = """
XGBoost treinado com tratamento de desbalanceamento
por scale_pos_weight e hiperparâmetros otimizados
com Optuna.

Modelo selecionado pela PR-AUC de validação e
avaliado pelas métricas de negócio Lift@K.

Uso principal:

Ranking mensal de clientes elegíveis para campanha
de reativação.
"""

client.update_model_version(
    name=(REGISTERED_MODEL_NAME),
    version=(model_version),
    description=(MODEL_DESCRIPTION),
)

MODEL_TAGS = {
    "project": "reactivation-customers-model",
    "model_family": "xgboost",
    "optimization": "optuna",
    "promotion_status": "approved",
    "primary_metric": METRIC_NAME,
    "source_run_id": best_run_id,
}

for tag_name, tag_value in MODEL_TAGS.items():
    client.set_model_version_tag(
        name=(REGISTERED_MODEL_NAME),
        version=(model_version),
        key=(tag_name),
        value=(str(tag_value)),
    )

In [0]:
client.set_registered_model_alias(
    name=(REGISTERED_MODEL_NAME),
    alias=("champion"),
    version=(model_version),
)


print(
    f"""
    Modelo aprovado.
    Alias:
    champion
    Versão:
    {model_version}
    """
)